In [1]:
from constant_iv_term_structure_engine import constantIVTermStructureEngine
import polars as pl

engine = constantIVTermStructureEngine(ticker="LITE", auto_sync=False)

# Step 1: Fetch equity data
equity_df = engine._fetch_equity_data()

# Step 2: Fetch options data
options_df = engine._fetch_options_data()

In [2]:
# Step 3: Filter options
filtered_options_df = engine._filter_options(options_df)

# Step 4: Add rolling equity metrics
equity_df = engine._add_equity_rolling_metrics(equity_df)

# Step 5: Filter equity
filtered_equity_df = engine._filter_equity(equity_df)

# Step 6: Add ATM diff, ATM IV, days_to_expiry: MIGHT BE THIS ONE
filtered_options_df = engine._add_options_derived_columns(filtered_options_df)

# # Step 7: Constant-maturity IV interpolation
cm_df = engine._compute_constant_maturity_iv(filtered_options_df)

# Step 8: Join options with constant-maturity data
options_cm_df = engine._join_options_with_cm(filtered_options_df, cm_df)

# Step 9: Join with equity data
consolidated_df = engine._join_with_equity(options_cm_df, filtered_equity_df)

In [3]:
# Step 10: Compute derived columns (SLOPE, IVRV_SLOPE, straddle prices)
consolidated_df1 = engine._compute_derived_columns(consolidated_df)

# Step 11: Final selection + wret
result = engine._select_final_and_compute_wret(consolidated_df1)

# Step 12: Add quantile ranking
result = engine.add_ntile(result, col=engine.rank_column, n=engine.n_tiles, by="trade_date")

In [6]:
result.filter(pl.col("cOpra") == "LITE260313C00667500")

ticker,expirDate,trade_date,cOpra,pOpra,cOpra_ST,pOpra_ST,cOpra_LT,pOpra_LT,cVolu,pVolu,cOi,pOi,min_leg_volume,total_volume,min_leg_oi,total_oi,dollar_vol,SLOPE,IVRV_SLOPE,straddle_price_bid,straddle_price_ask,straddle_price_mid,straddle_price_bid_ST,straddle_price_ask_ST,straddle_price_mid_ST,spread_abs,spread_pct_mid,wret,q_slope
str,datetime[μs],date,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""LITE""",2026-03-13 00:00:00,2026-03-11,"""LITE260313C00667500""","""LITE260313P00667500""","""LITE260410C00665000""","""LITE260410P00665000""","""LITE260918C00680000""","""LITE260918P00680000""",5.0,6.0,367.0,19.0,5.0,11.0,19.0,386.0,8.1000e10,0.002908,-0.220534,43.5,53.5,48.5,150.2,162.5,156.35,10.0,0.206186,null,10


In [5]:
result.filter(pl.col("trade_date") == pl.col("trade_date").max())

ticker,expirDate,trade_date,cOpra,pOpra,cOpra_ST,pOpra_ST,cOpra_LT,pOpra_LT,cVolu,pVolu,cOi,pOi,min_leg_volume,total_volume,min_leg_oi,total_oi,dollar_vol,SLOPE,IVRV_SLOPE,straddle_price_bid,straddle_price_ask,straddle_price_mid,straddle_price_bid_ST,straddle_price_ask_ST,straddle_price_mid_ST,spread_abs,spread_pct_mid,wret,q_slope
str,datetime[μs],date,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""LITE""",2026-03-13 00:00:00,2026-03-11,"""LITE260313C00667500""","""LITE260313P00667500""","""LITE260410C00665000""","""LITE260410P00665000""","""LITE260918C00680000""","""LITE260918P00680000""",5.0,6.0,367.0,19.0,5.0,11.0,19.0,386.0,8.1000e10,0.002908,-0.220534,43.5,53.5,48.5,150.2,162.5,156.35,10.0,0.206186,null,10
